In [0]:
spark.version

#Getting Exising Cluster ID(Existing Clster)

In [0]:
spark.conf.get("spark.databricks.clusterUsageTags.clusterId")

#Droping tables if exist

In [0]:
spark.sql(f" drop table if exists bronze.shows")
spark.sql(f" drop table if exists bronze.episodes")
spark.sql(f" drop table if exists bronze.cast" )

In [0]:
spark.sql(f" drop schema if exists bronze.shows")
spark.sql(f" drop schema if exists bronze.episodes")
spark.sql(f" drop schema if exists bronze.cast" )

Prepare schema for show dataframe/table

In [0]:
from pyspark.sql.types import *

# Define nested schemas FIRST
schedule_schema = StructType([
    StructField("time", StringType(), True),
    StructField("days", ArrayType(StringType()), True)
])

rating_schema = StructType([
    StructField("average", DoubleType(), True)
])

network_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("country", StructType([
        StructField("name", StringType(), True),
        StructField("code", StringType(), True),
        StructField("timezone", StringType(), True)
    ]), True)
])


In [0]:
from pyspark.sql.types import *
shows_schema = StructType([
    StructField("id", LongType(), True),
    StructField("url", StringType(), True),
    StructField("name", StringType(), True),
    StructField("type", StringType(), True),
    StructField("language", StringType(), True),
    StructField("genres", ArrayType(StringType()), True),
    StructField("status", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("averageRuntime", IntegerType(), True),
    StructField("premiered", StringType(), True),
    StructField("ended", StringType(), True),
    StructField("officialSite", StringType(), True),
    StructField("schedule", schedule_schema, True),
    StructField("rating", rating_schema, True),
    StructField("weight", IntegerType(), True),
    StructField("network", network_schema, True)
])


Extracting data from the given/specified web url or API

In [0]:
import requests

url = "https://api.tvmaze.com/shows"
data = requests.get(url).json()

shows_df = spark.createDataFrame(data, schema=shows_schema)
display(shows_df)

Prepare Schema for TV Maze episode Dataframe/table

In [0]:
from pyspark.sql.types import *

# Nested schemas
rating_schema = StructType([
    StructField("average", IntegerType(), True)
])

image_schema = StructType([
    StructField("medium", StringType(), True),
    StructField("original", StringType(), True)
])

links_self_schema = StructType([
    StructField("href", StringType(), True)
])

links_show_schema = StructType([
    StructField("href", StringType(), True),
    StructField("name", StringType(), True)
])

links_schema = StructType([
    StructField("self", links_self_schema, True),
    StructField("show", links_show_schema, True)
])

# Main episode schema
episode_schema = StructType([
    StructField("id", LongType(), True),
    StructField("url", StringType(), True),
    StructField("name", StringType(), True),
    StructField("season", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("airdate", StringType(), True),
    StructField("airtime", StringType(), True),
    StructField("airstamp", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("rating", rating_schema, True),
    StructField("image", image_schema, True),
    StructField("summary", StringType(), True),
    StructField("_links", links_schema, True)
])


In [0]:
episode_schema = StructType([
    StructField("id", LongType(), True),          # episode id
    StructField("show_id", LongType(), True),     # ✅ REQUIRED (FK)
    StructField("url", StringType(), True),
    StructField("name", StringType(), True),
    StructField("season", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("airdate", StringType(), True),
    StructField("airtime", StringType(), True),
    StructField("airstamp", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("rating", rating_schema, True),
    StructField("image", image_schema, True),
    StructField("summary", StringType(), True),
    StructField("_links", links_schema, True)
])


Extracting Episodes data from given/Specified TV Maze web url or API

In [0]:
import requests

show_id = shows_df.select("id").collect()[0]["id"]   # example
url = f"https://api.tvmaze.com/shows/{show_id}/episodes"

episodes_data = requests.get(url).json()


In [0]:
episodes_with_show_id = []

for ep in episodes_data:
    ep["show_id"] = show_id
    episodes_with_show_id.append(ep)

In [0]:
show_ids = [row.id for row in shows_df.select("id").collect()];
show_ids 

In [0]:
def get_episodes(show_id):
    url = f"https://api.tvmaze.com/shows/{show_id}/episodes"
    response = requests.get(url)

    data = response.json()
    return data if isinstance(data, list) else []

In [0]:
all_episodes = []

for show_id in show_ids:
    episodes = get_episodes(show_id)

    for ep in episodes:
        ep["show_id"] = show_id   # ✅ key relationship
        all_episodes.append(ep)

Extracting Episodes data 

In [0]:
episodes_df = spark.createDataFrame(
    all_episodes,
    schema=episode_schema
)

display(episodes_df)

In [0]:
shows_episodes_df = (
    episodes_df
    .join(
        shows_df.select("id", "name"),
        episodes_df.show_id == shows_df.id,
        "inner"
    )
    .drop(shows_df.id)
    .withColumnRenamed("name", "show_name")
)

display(shows_episodes_df)

prepare Schema for CAST Dataframe/Table:

In [0]:
from pyspark.sql.types import *

person_schema = StructType([
    StructField("id", LongType(), True),
    StructField("url", StringType(), True),
    StructField("name", StringType(), True),
    StructField("country", StructType([
        StructField("name", StringType(), True),
        StructField("code", StringType(), True),
        StructField("timezone", StringType(), True)
    ]), True),
    StructField("birthday", StringType(), True),
    StructField("deathday", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("image", StructType([
        StructField("medium", StringType(), True),
        StructField("original", StringType(), True)
    ]), True)
])

character_schema = StructType([
    StructField("id", LongType(), True),
    StructField("url", StringType(), True),
    StructField("name", StringType(), True),
    StructField("image", StructType([
        StructField("medium", StringType(), True),
        StructField("original", StringType(), True)
    ]), True)
])


In [0]:
cast_schema = StructType([
    StructField("show_id", LongType(), True),     # ✅ FK
    StructField("person", person_schema, True),
    StructField("character", character_schema, True),
    StructField("self", BooleanType(), True),
    StructField("voice", BooleanType(), True)
])


In [0]:
show_ids = [row.id for row in shows_df.select("id").collect()]

In [0]:
import requests

def get_cast(show_id):
    url = f"https://api.tvmaze.com/shows/{show_id}/cast"
    response = requests.get(url)

    data = response.json()
    return data if isinstance(data, list) else []

In [0]:
all_cast = []

for show_id in show_ids:
    cast_list = get_cast(show_id)

    for cast in cast_list:
        cast["show_id"] = show_id   # ✅ link to show
        all_cast.append(cast)

Extracting CAST Data from the given/Specified web url or API

In [0]:

cast_df = spark.createDataFrame(
    all_cast,
    schema=cast_schema
)

display(cast_df)

In [0]:

shows_cast_df = (
    cast_df
    .join(
        shows_df.select("id", "name"),
        cast_df.show_id == shows_df.id,
        "inner"
    )
    .drop(shows_df.id)
    .withColumnRenamed("name", "show_name")
)

display(shows_cast_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

##Exporting/ Saving DataFrames as Delta Tables

In [0]:
shows_df.write.format("delta").option("mergeSchema", "true").option("overwriteSchema", "true").mode("overwrite").saveAsTable("bronze.shows")
episodes_df.write.format("delta").option("mergeSchema", "true").option("overwriteSchema", "true").mode("overwrite").saveAsTable("bronze.episodes")
cast_df.write.format("delta").option("mergeSchema", "true").option("overwriteSchema", "true").mode("overwrite").saveAsTable("bronze.cast")

#Getting count for all 3 Dataframes/Tables

In [0]:
df = spark.sql(f"""
SELECT 'shows' as label, COUNT(*) as count FROM bronze.shows
union all
SELECT 'episodes' as label, COUNT(*) as count FROM bronze.episodes
union all
SELECT 'cast' as label, COUNT(*) as count FROM bronze.cast
""")
display(df)